# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irssaa29/Machine-learning_01/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
This is a ranking/scoring problem, not classification. The business question isn't "will this page decline?" (yes/no) — it's "which pages should be reviewed first, given limited reviewer time?" That's a priority ordering, which means the output is a ranked list, not a single label. Ranking/scoring is also what Week 1's Precision@K comparison (hand rule vs. tree) was already testing, so this task type is a natural continuation of that work.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
My target is a defined proxy called priority_target: a page is high-priority if it is (1) declining (is_declining_label == 1), (2) still visible (impressions_90d above a threshold), and (3) old enough that staleness is a plausible cause (content_age_days above a threshold). I chose this instead of is_declining_label alone because a declining page nobody sees isn't worth a reviewer's time — combining these three signals gets closer to genuine priority.

Important honesty check: is_declining_label is itself computed from trend_direction, which comes from trend_pct — so my target is a defined proxy, not a fully observed outcome. A truly observed target would track whether a page's traffic actually recovered after a reviewer acted on it, which this dataset doesn't capture. Because of this, trend_direction and trend_pct can only ever be used to build the target — never as features feeding the model, or the model would just be learning to detect its own label (leakage).

## 3. Success metric

*One metric you can defend. What number means 'good'?*
I'll use Precision@K — of the top K pages my ranking flags, what fraction are true priority_target == 1? This is the same metric I used in Week 1, so I can directly compare how a more nuanced target changes performance versus the simpler is_declining_label I scored before. I'll check K=20 and K=50, consistent with prior work.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/irssaa29/Machine-learning_01"
REPO_DIR = "Machine-learning_01"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

df["priority_target"] = (
    (df["is_declining_label"] == 1) &
    (df["impressions_90d"] >= 100) &
    (df["content_age_days"] >= 90)
).astype(int)

print(df.shape)
print("Priority target rate:", round(df["priority_target"].mean(), 3))
df[["content_id", "is_declining_label", "impressions_90d", "content_age_days", "priority_target"]].head(10)


(30000, 46)
Priority target rate: 0.438


,content_id,is_declining_label,impressions_90d,content_age_days,priority_target
0,content_304f48230142,1,3803,187,1
1,content_a1fb4e703a9e,1,15320,445,1
2,content_9aa793d4d895,1,12581,141,1
3,content_331d6c4de07b,0,11751,463,0
4,content_d99b7a2d90ca,1,19140,263,1
5,content_d4084a4bc775,1,3970,147,1
6,content_9a34b442b552,1,20,90,0
7,content_a63219c6e95a,0,1724,445,0
8,content_5e6c160719bc,1,32574,90,1
9,content_c27558df2b0c,1,1240,257,1


One row = one content page (content_id), with 30,000 pages in this slice, 43.8% flagged as priority.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
A fixed rule could technically compute priority_target directly (it's literally three thresholds combined) — but the thresholds themselves are arbitrary guesses (100 impressions? 90 days?), and real priority likely depends on interacting, non-linear combinations of many more signals (position, CTR, word count, freshness tier) that a human can't hand-tune reliably. In Week 1, my strict hand rule ("stale AND visible") only caught 17 pages out of 30,000 — far too narrow to be useful — while a decision tree found a similar-quality ranking using softer, data-derived thresholds. That's the case for ML here: the pattern is real, but too messy and multi-signal to write by hand with confidence.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.